In [ ]:
from selenium import webdriver # Navegador
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import undetected_chromedriver as uc


In [ ]:
import pandas as pd
import sys
import shutil
import os
import time
import datetime
import pickle


In [ ]:
# Setup Chrome options
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
options.add_argument("--window-size=1920x1080")

# options.add_argument("--headless")  # Optional: hide the browser

# This automatically finds the correct version of ChromeDriver
driver = uc.Chrome(options=options)

# Add any options you need, e.g.:
# chrome_options.add_argument("--headless")

# Allow automatic downloads to a custom path
driver.command_executor._commands["send_command"] = ("POST", '/session/$sessionId/chromium/send_command')

# Set timeouts
driver.implicitly_wait(2)
driver.set_script_timeout(900)
driver.set_page_load_timeout(900)
driver.maximize_window()
driver.get("https://www.immobiliare.it/")


In [ ]:
luogo = 'Roma'

prezzo_min = None
prezzo_max = 200000

superficie_min = 40
superficie_max = None

In [ ]:
# Step 1: Click the wrapper div to activate the input field
location_click_target = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "/html/body/div[2]/div[1]/div/div/div[2]/div/div[1]/div"))
)
location_click_target.click()

# Step 2: Find the input field
location_input = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, "/html/body/div[2]/div[1]/div/div/div[2]/div/div[1]/div/div[1]/div/input"))
)

# Step 3: Type the city name (e.g., Roma)
location_input.send_keys(luogo)
time.sleep(10)

In [ ]:
location_input.send_keys(Keys.ENTER)

In [ ]:
# Wait for the accordion container to load
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "in-placeCheckListAccordion"))
)

# Find all macrozone containers
macrozones = driver.find_elements(By.XPATH, "//div[contains(@class, 'nd-accordion__item')]")

print("Number of macrozones found:", len(macrozones))

In [ ]:
numero_zone = len(macrozones)
numero_zone

In [ ]:
def toggle_macrozone(driver, macro_zona):
    xpath_button = f"""//div[@role='button' and contains(@class, 'nd-accordion__title')]
                       [descendant::span[contains(normalize-space(), "{macro_zona}")]]"""

    accordion_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, xpath_button))
    )

    # Scroll to it
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", accordion_button)
    time.sleep(0.5)

    # Get current state
    current_state = accordion_button.get_attribute("aria-expanded")

    if current_state == "false":
        print(f"🔽 Expanding macrozona: {macro_zona}")
        driver.execute_script("arguments[0].click();", accordion_button)
        WebDriverWait(driver, 5).until(
            lambda d: accordion_button.get_attribute("aria-expanded") == "true"
        )
    elif current_state == "true":
        print(f"🔼 Collapsing macrozona: {macro_zona}")
        driver.execute_script("arguments[0].click();", accordion_button)
        WebDriverWait(driver, 5).until(
            lambda d: accordion_button.get_attribute("aria-expanded") == "false"
        )
    else:
        print(f"⚠️ Unknown state for {macro_zona}: {current_state}")

In [ ]:
dict_zone = dict()
for i in range(2,numero_zone+1):
    macro_zona_button = driver.find_element(By.XPATH, '/html/body/div[2]/div[1]/div/div/div[2]/div/div[1]/div/div/div[2]/div[2]/div[{}]/div[1]/div/label/span'.format(i))
    macro_zona = macro_zona_button.text
    # XPath: find the accordion title containing that macrozone name
    # XPath to get the accordion button for this macrozona
    toggle_macrozone(driver, macro_zona)   
    
    xpath_template = '/html/body/div[2]/div[1]/div/div/div[2]/div/div[1]/div/div/div[2]/div[2]/div[{}]/div[2]/div/ul/li'
    n_sub_elements = len(driver.find_elements(By.XPATH, xpath_template.format(i)))    
    print(n_sub_elements)
    for j in range(1, n_sub_elements+1):
        sub_obj = driver.find_element(By.XPATH,'/html/body/div[2]/div[1]/div/div/div[2]/div/div[1]/div/div/div[2]/div[2]/div[{}]/div[2]/div/ul/li[{}]/label/span'.format(i,j))
        nome_sottozona = sub_obj.get_attribute("textContent").strip()
        print(nome_sottozona)
        dict_zone[nome_sottozona] = (i,j)
        html = driver.find_element(By.TAG_NAME,'html')
        html.send_keys(Keys.DOWN)
        time.sleep(1)
    count=0
    while count<n_sub_elements:
        html = driver.find_element(By.TAG_NAME,'html')
        html.send_keys(Keys.UP)
        time.sleep(1)
        count+=1
    time.sleep(5)
    toggle_macrozone(driver, macro_zona)
    
    html = driver.find_element(By.TAG_NAME, 'html')
    html.send_keys(Keys.DOWN)


In [ ]:
with open('dizionario_zone_roma.pickle', 'wb') as handle:
    pickle.dump(dict_zone, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [ ]:
dict_zone